# Text EDA: documentation corpus and P1 evaluation set
ADSP 32018 | corpus_v1, eval v1

## 0. Setup

In [ ]:
import json, re, collections, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

CORPUS = Path("corpus/corpus_v1.jsonl")
EVAL = Path("eval/qa_eval.jsonl")
FIG, TAB = Path("figures"), Path("tables")
FIG.mkdir(exist_ok=True); TAB.mkdir(exist_ok=True)
SEED = 42
np.random.seed(SEED)

corpus = pd.read_json(CORPUS, lines=True)
corpus["component"] = corpus.metadata.apply(lambda m: m.get("component"))
corpus["omop_table"] = corpus.metadata.apply(lambda m: m.get("table"))
corpus["group"] = corpus.apply(lambda r: r.source + " / " + r.kind.replace(r.source + "_", "", 1), axis=1)
print(corpus.shape)

## 1. Data summary

In [ ]:
summary = (corpus.groupby(["source", "kind"])
           .agg(records=("id", "count"), docs=("doc", "nunique")).reset_index())
summary.loc[len(summary)] = ["TOTAL", "", len(corpus), corpus.doc.nunique()]
summary.to_csv(TAB / "corpus_summary.csv", index=False)
summary

## 2. Text statistics

In [ ]:
TOKEN = re.compile(r"[A-Za-z0-9_]+")
corpus["chars"] = corpus.text.str.len()
corpus["tokens"] = corpus.text.apply(lambda t: len(TOKEN.findall(t)))
corpus["lines"] = corpus.text.str.count("\n") + 1

stats = (corpus.groupby("group")[["chars", "tokens"]]
         .describe(percentiles=[.5, .9]).round(1))
stats.columns = ["_".join(c) for c in stats.columns]
stats.to_csv(TAB / "length_stats_by_group.csv")
stats[["chars_count", "chars_50%", "chars_90%", "chars_max", "tokens_50%", "tokens_90%", "tokens_max"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
order = corpus.groupby("group").tokens.median().sort_values().index
sns.boxplot(data=corpus, y="group", x="tokens", order=order, ax=axes[0], showfliers=False)
axes[0].set(xlabel="tokens per chunk (outliers hidden)", ylabel="")
for g in order:
    sns.ecdfplot(np.log10(corpus.loc[corpus.group == g, "tokens"]), ax=axes[1], label=g)
axes[1].set(xlabel="log10(tokens per chunk)", ylabel="cumulative share")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "chunk_length_distributions.png", dpi=160); plt.show()

limits = [256, 512, 1024]
pd.DataFrame({f">{l} tokens": corpus.groupby("group").tokens.apply(lambda s: (s > l).mean()).round(3)
              for l in limits})

In [ ]:
CRYPTIC = re.compile(r"^(?:[A-Z]{2,}[A-Z0-9_]*|[A-Za-z]+_[A-Za-z0-9_]+|[A-Za-z]+\d+[A-Za-z0-9]*)$")
def cryptic_share(texts):
    toks = [t for x in texts for t in TOKEN.findall(x)]
    return sum(bool(CRYPTIC.match(t)) for t in toks) / max(len(toks), 1), len(toks)

rows = []
for src, g in list(corpus.groupby("source")) + [("ALL", corpus)]:
    toks = [t.lower() for x in g.text for t in TOKEN.findall(x)]
    share, n = cryptic_share(g.text)
    rows.append(dict(source=src, tokens=n, vocab=len(set(toks)),
                     type_token_ratio=round(len(set(toks)) / n, 4),
                     cryptic_identifier_share=round(share, 4)))
vocab = pd.DataFrame(rows)
vocab.to_csv(TAB / "vocab_and_cryptic_share.csv", index=False)
vocab

## 3. Linguistic features

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
def top_ngrams(texts, n=(2, 2), k=15):
    cv = CountVectorizer(ngram_range=n, stop_words="english", token_pattern=r"(?u)\b[A-Za-z][A-Za-z_]+\b",
                         lowercase=True, min_df=2)
    X = cv.fit_transform(texts)
    freq = np.asarray(X.sum(axis=0)).ravel()
    idx = freq.argsort()[::-1][:k]
    return pd.Series(freq[idx], index=cv.get_feature_names_out()[idx])

groups = ["nhanes / variable", "nhanes / narrative", "omop / field", "mimic / section"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ngram_rows = []
for ax, g in zip(axes.ravel(), groups):
    s = top_ngrams(corpus.loc[corpus.group == g, "text"])
    s[::-1].plot.barh(ax=ax); ax.set_title(g)
    ngram_rows += [(g, w, c) for w, c in s.items()]
plt.tight_layout(); plt.savefig(FIG / "top_bigrams.png", dpi=160); plt.show()
pd.DataFrame(ngram_rows, columns=["group", "bigram", "count"]).to_csv(TAB / "top_bigrams.csv", index=False)

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
sample = pd.concat([g.sample(min(len(g), 120), random_state=SEED) for _, g in corpus.groupby("group")])
docs = list(nlp.pipe(sample.text.str.slice(0, 1500), batch_size=32))

pos_rows, ent_rows = [], []
for grp, d in zip(sample.group, docs):
    for t in d:
        if not t.is_space and not t.is_punct:
            pos_rows.append((grp, t.pos_))
    for e in d.ents:
        ent_rows.append((grp, e.label_, e.text))
pos = pd.DataFrame(pos_rows, columns=["group", "pos"])
pos_share = pos.groupby("group").pos.value_counts(normalize=True).unstack(fill_value=0)
keep = pos_share.mean().sort_values(ascending=False).head(10).index
pos_share[keep].round(3).to_csv(TAB / "pos_share.csv")
pos_share[keep].plot.bar(stacked=True, figsize=(10, 4.5), colormap="tab20")
plt.ylabel("share of tokens"); plt.legend(bbox_to_anchor=(1, 1)); plt.tight_layout()
plt.savefig(FIG / "pos_distribution.png", dpi=160); plt.show()
pos_share[keep].round(3)

In [ ]:
ents = pd.DataFrame(ent_rows, columns=["group", "label", "text"])
ent_counts = ents.groupby(["group", "label"]).size().unstack(fill_value=0)
ent_counts.to_csv(TAB / "entity_counts.csv")
top_ents = pd.concat([g.groupby(["label", "text"]).size().sort_values(ascending=False).head(5)
                      .reset_index(name="n").assign(group=name) for name, g in ents.groupby("group")])
top_ents.to_csv(TAB / "top_entities.csv", index=False)
ent_counts

In [ ]:
CODE_ROW = re.compile(r"^(\d{1,5}|\.)\s*\|\s*([^|]+?)\s*\|", re.M)
def categorize(desc):
    d = desc.lower()
    if "refus" in d: return "refused"
    if "don't know" in d or "dont know" in d: return "don't know"
    if d.startswith("missing"): return "missing"
    if re.search(r"\d+\s*(to|-)\s*\d+|range of values|or more|and over|less than|greater than|above|below", d): return "range / bound"
    return "substantive value"

code_rows = []
for _, r in corpus[corpus.kind == "nhanes_variable"].iterrows():
    for val, desc in CODE_ROW.findall(r.text):
        code_rows.append(dict(variable=r.id, component=r.component, value=val, desc=desc, category=categorize(desc)))
codes = pd.DataFrame(code_rows)

per_var = corpus[corpus.kind == "nhanes_variable"].assign(
    has_code_table=lambda d: d.text.str.contains("Code or Value"),
    has_refused=lambda d: d.text.str.contains("Refused"),
    has_dont_know=lambda d: d.text.str.contains("Don't know"),
    has_skip=lambda d: d.text.str.contains(r"\| [A-Z]{2,}\w*\s*$", regex=True))
print("variables:", len(per_var))
print(per_var[["has_code_table", "has_refused", "has_dont_know"]].mean().round(3).to_dict())

cat_counts = codes.category.value_counts()
cat_counts.to_csv(TAB / "value_code_categories.csv")
code_values = (codes[codes.category.isin(["refused", "don't know"])]
               .groupby(["category", "value"]).size().rename("variables").reset_index())
code_values.to_csv(TAB / "refusal_code_values.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
cat_counts.plot.barh(ax=ax[0]); ax[0].set_title("code-table row categories")
(per_var.groupby("component")[["has_refused", "has_dont_know"]].mean()
 .sort_values("has_refused").plot.barh(ax=ax[1])); ax[1].set_title("share of variables with 7 / 9 codes")
plt.tight_layout(); plt.savefig(FIG / "value_code_patterns.png", dpi=160); plt.show()
code_values

## 4. Visualizations

In [ ]:
from wordcloud import WordCloud
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
stop = set(ENGLISH_STOP_WORDS) | {"nhanes", "variable", "name", "sas", "label", "english", "text", "target", "codes", "code", "value", "description", "count", "skip", "item", "missing"}
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, g in zip(axes.ravel(), groups):
    text = " ".join(corpus.loc[corpus.group == g, "text"])
    wc = WordCloud(width=700, height=400, background_color="white", stopwords=stop,
                   random_state=SEED, collocations=False).generate(text)
    ax.imshow(wc, interpolation="bilinear"); ax.set_title(g); ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "wordclouds.png", dpi=160); plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

texts = corpus.text.str.slice(0, 2000).tolist()
try:
    from sentence_transformers import SentenceTransformer
    EMB_NAME = "all-MiniLM-L6-v2"
    model = SentenceTransformer(EMB_NAME)
    E = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
except Exception as e:
    EMB_NAME = "tfidf-svd128 (fallback)"
    tf = TfidfVectorizer(sublinear_tf=True, min_df=2, max_features=30000, ngram_range=(1, 2))
    E = normalize(TruncatedSVD(128, random_state=SEED).fit_transform(tf.fit_transform(texts)))
print("embedding:", EMB_NAME, E.shape)
np.save("embeddings.npy", E)

In [ ]:
import umap
XY = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED).fit_transform(E)
corpus["u1"], corpus["u2"] = XY[:, 0], XY[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(data=corpus, x="u1", y="u2", hue="source", s=14, ax=axes[0], linewidth=0)
axes[0].set_title("UMAP by source (" + EMB_NAME + ")")
nh = corpus[corpus.source == "nhanes"]
sns.scatterplot(data=nh, x="u1", y="u2", hue="component", s=18, ax=axes[1], palette="tab20", linewidth=0)
axes[1].set_title("NHANES chunks by component"); axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(FIG / "umap_source_component.png", dpi=160); plt.show()

In [ ]:
nh_var = corpus[corpus.kind == "nhanes_variable"].index
omop_tab = corpus[corpus.source == "omop"]
E_df = pd.DataFrame(E, index=corpus.index)

comp_centroid = E_df.loc[nh_var].groupby(corpus.loc[nh_var, "component"]).mean()
tab_centroid = E_df.loc[omop_tab.index].groupby(omop_tab.omop_table).mean()
top_tables = ["person", "visit_occurrence", "condition_occurrence", "measurement", "observation",
              "drug_exposure", "procedure_occurrence", "death", "specimen", "device_exposure"]
top_tables = [t for t in top_tables if t in tab_centroid.index]
A = normalize(comp_centroid.values) @ normalize(tab_centroid.loc[top_tables].values).T
heat = pd.DataFrame(A, index=comp_centroid.index, columns=top_tables)
heat.round(3).to_csv(TAB / "component_vs_omop_table_similarity.csv")
plt.figure(figsize=(10, 6))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="viridis", cbar_kws={"label": "cosine similarity of centroids"})
plt.title("NHANES component vs OMOP table (" + EMB_NAME + ")"); plt.tight_layout()
plt.savefig(FIG / "similarity_heatmap_component_omop.png", dpi=160); plt.show()

In [ ]:
rows = []
for _, r in corpus[corpus.kind == "nhanes_variable"].iterrows():
    m_lab = re.search(r"SAS Label:\s*(.+)", r.text); m_eng = re.search(r"English Text:\s*(.+)", r.text)
    if m_lab and m_eng:
        rows.append(dict(id=r.id, component=r.component, label=m_lab.group(1).strip(), english=m_eng.group(1).strip()))
pairs = pd.DataFrame(rows)
if EMB_NAME.startswith("all-MiniLM"):
    L = model.encode(pairs.label.tolist(), normalize_embeddings=True, show_progress_bar=False)
    T = model.encode(pairs.english.tolist(), normalize_embeddings=True, show_progress_bar=False)
else:
    tf2 = TfidfVectorizer(sublinear_tf=True, min_df=1).fit(pairs.label.tolist() + pairs.english.tolist())
    svd = TruncatedSVD(64, random_state=SEED).fit(tf2.transform(pairs.label.tolist() + pairs.english.tolist()))
    L = normalize(svd.transform(tf2.transform(pairs.label.tolist())))
    T = normalize(svd.transform(tf2.transform(pairs.english.tolist())))
pairs["sim"] = (L * T).sum(axis=1)
pairs.sort_values("sim").head(15).to_csv(TAB / "lowest_label_vs_text_alignment.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(pairs.sim, bins=30, ax=ax[0]); ax[0].set(xlabel="cosine(SAS label, English text)")
sns.boxplot(data=pairs, y="component", x="sim", ax=ax[1]); ax[1].set(xlabel="cosine(SAS label, English text)", ylabel="")
plt.tight_layout(); plt.savefig(FIG / "label_vs_question_alignment.png", dpi=160); plt.show()
pairs.sort_values("sim").head(10)[["id", "label", "english", "sim"]]

## 5. P1 evaluation set inspection

In [ ]:
ev = pd.read_json(EVAL, lines=True)
ev["q_tokens"] = ev.question.apply(lambda t: len(TOKEN.findall(t)))
ev["a_tokens"] = ev.answer.apply(lambda t: len(TOKEN.findall(t)))
print(ev.shape)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), ["qtype", "source", "split", "needs_data", "guessable"]):
    ev[col].value_counts().plot.bar(ax=ax); ax.set_title(col); ax.tick_params(axis="x", rotation=0)
sns.histplot(data=ev, x="q_tokens", hue="qtype", multiple="stack", ax=axes[1, 2], bins=20)
plt.tight_layout(); plt.savefig(FIG / "eval_set_composition.png", dpi=160); plt.show()
pd.crosstab([ev.qtype], [ev.source, ev.split]).to_csv(TAB / "eval_qtype_source_split.csv")
pd.crosstab([ev.qtype], [ev.source, ev.split])

In [ ]:
doc_ids = set(corpus.doc)
ev["gold_doc_in_corpus"] = ev.gold_doc.isin(doc_ids)
cited_sections = ev.dropna(subset=["gold_doc", "gold_section"])
cov = pd.Series({
    "eval questions": len(ev),
    "gold_doc found in corpus": int(ev.gold_doc_in_corpus.sum()),
    "distinct gold docs": ev.gold_doc.nunique(),
    "corpus docs cited": f"{ev.gold_doc[ev.gold_doc_in_corpus].nunique()} / {corpus.doc.nunique()}",
    "guessable == yes": int((ev.guessable == "yes").sum()),
    "needs_data == yes": int((ev.needs_data == "yes").sum()),
})
cov.to_csv(TAB / "eval_coverage.csv", header=["value"])
print(cov.to_string())
ev.groupby("qtype")[["q_tokens", "a_tokens"]].median()

In [ ]:
ev_rows = ev.sample(min(8, len(ev)), random_state=SEED)[["qid", "qtype", "source", "question", "answer"]]
ev_rows.to_csv(TAB / "eval_sample_rows.csv", index=False)
ev_rows